# Real-Time Card-Not-Present Fraud Detection using SVM
### FinTech / E-Commerce Payments — SVM Kernel Comparison, Tuning, Feature Selection & Evaluation

**Dataset:** Credit Card Fraud Detection (anonymized PCA features)
**Source:** Kaggle — `mlg-ulb/creditcardfraud`
**URL:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
**Version:** As available on Kaggle (284,807 transactions, 31 columns: Time, Amount, V1–V28 (PCA-transformed), Class)

**Author:** Amrita Jadhav

---
### Objective
Build a Support Vector Machine (SVM) based fraud classifier for card-not-present transactions.
We compare linear vs. non-linear kernels, tune hyperparameters, apply feature selection, and
evaluate using metrics appropriate for a highly imbalanced, rare-event classification problem.


## Step 1: Setup

Install any non-standard packages here so the notebook runs top-to-bottom without manual intervention.


In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              precision_recall_curve, auc, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42


## Step 2: Load the Dataset

Upload `creditcard.csv` to this Colab session before running this cell
(Files panel on the left → Upload → select the file), or mount Google Drive and point to its path.


In [ ]:
# If using Colab file upload:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('creditcard.csv')
print("Shape:", df.shape)
df.head()


## Step 3: Exploratory Data Analysis (EDA)

### 3.1 Basic structure & missing values


In [ ]:
print(df.info())
print("\nMissing values per column:\n", df.isnull().sum().sum(), "total missing values")


### 3.2 Class Imbalance

This is the central challenge of this dataset. Fraud is rare, so we must check the exact
imbalance ratio — this drives every later decision (stratified splitting, stratified CV,
using PR-AUC instead of plain accuracy).


In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("Class counts:\n", class_counts)
print("\nClass percentage:\n", class_pct)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Class', data=df, ax=ax[0])
ax[0].set_title('Class Distribution (Count)')
ax[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

ax[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%', colors=['#4C72B0', '#C44E52'])
ax[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.show()


**Observation to write in your own words after running:** Note the exact fraud percentage
(usually ~0.17% for this dataset). This confirms fraud detection is a rare-event problem, meaning
plain accuracy would be misleading (a model predicting "always legitimate" would still score ~99.8%
accuracy while catching zero fraud).


### 3.3 Amount & Time — Fraud vs Legitimate


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Class', y='Amount', data=df, ax=ax[0], showfliers=False)
ax[0].set_title('Transaction Amount by Class (outliers hidden for scale)')
ax[0].set_xticklabels(['Legitimate', 'Fraud'])

sns.histplot(df[df['Class']==0]['Time'], bins=50, color='#4C72B0', label='Legitimate', alpha=0.5, ax=ax[1])
sns.histplot(df[df['Class']==1]['Time'], bins=50, color='#C44E52', label='Fraud', alpha=0.5, ax=ax[1])
ax[1].set_title('Transaction Time Distribution by Class')
ax[1].legend()

plt.tight_layout()
plt.show()

print(df.groupby('Class')['Amount'].describe())


### 3.4 Separability of PCA Features (V1–V4) by Class

The dataset's V1–V28 features are already PCA-transformed. We check a few of them visually to
see whether fraud and legitimate transactions form separable clusters — this tells us whether
a **linear** decision boundary is likely to be enough, or whether we need a non-linear kernel.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features_to_check = ['V1', 'V2', 'V3', 'V4']

for i, feat in enumerate(features_to_check):
    ax = axes[i // 2, i % 2]
    sns.kdeplot(df[df['Class']==0][feat], label='Legitimate', ax=ax, fill=True, alpha=0.3)
    sns.kdeplot(df[df['Class']==1][feat], label='Fraud', ax=ax, fill=True, alpha=0.3)
    ax.set_title(f'Distribution of {feat} by Class')
    ax.legend()

plt.tight_layout()
plt.show()


**Observation to write after running:** If the fraud and legitimate distributions overlap
heavily for most features but separate somewhat for a few (commonly V4, V11, V12, V14, V17 in this
dataset), that's a signal the classes are **not cleanly linearly separable** — motivating our
comparison of linear vs. RBF vs. polynomial kernels in the next step.


## Step 4: Preprocessing

### 4.1 Scale `Amount` and `Time`

SVM is a distance-based algorithm — unscaled features with large ranges (like `Amount`) would
dominate the margin calculation. `V1`–`V28` are already PCA-transformed and roughly standardized,
so we only need to scale `Amount` and `Time`.


In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

df_model = df.drop(['Amount', 'Time'], axis=1)

X = df_model.drop('Class', axis=1)
y = df_model['Class']

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())


### 4.2 Stratified Train-Test Split

We use `stratify=y` so that the ~0.17% fraud ratio is preserved identically in both the
training and test sets. Without this, a random split could leave the test set with very few
(or zero) fraud examples, making evaluation unreliable.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " Fraud ratio:", y_train.mean())
print("Test shape:", X_test.shape, " Fraud ratio:", y_test.mean())


## Step 5: Baseline SVM — Linear Kernel

We start with the simplest kernel as a benchmark. This establishes a baseline that RBF and
polynomial kernels must beat to justify their added complexity and training cost.

**Note:** SVM training scales poorly with dataset size (roughly O(n^2) to O(n^3)). Training on
all ~227K rows would be very slow, especially once we get to kernel comparison and grid search.
We'll use a **stratified subsample** (keeping the same fraud ratio) for the kernel-comparison and
tuning stages, and note this as a deliberate engineering tradeoff in our final writeup.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Create a stratified subsample (~20,000 rows) for faster kernel comparison & tuning
sss = StratifiedShuffleSplit(n_splits=1, train_size=20000, random_state=RANDOM_STATE)
sub_idx, _ = next(sss.split(X_train, y_train))

X_train_sub = X_train.iloc[sub_idx]
y_train_sub = y_train.iloc[sub_idx]

print("Subsample shape:", X_train_sub.shape, " Fraud ratio:", y_train_sub.mean())


In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    if hasattr(model, "decision_function"):
        y_scores = model.decision_function(X_test)
    else:
        y_scores = model.predict_proba(X_test)[:, 1]

    prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_scores)
    pr_auc = auc(rec_curve, prec_curve)

    print(f"--- {model_name} ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print()

    return {"model": model_name, "precision": precision, "recall": recall,
            "f1": f1, "pr_auc": pr_auc}


In [ ]:
start = time.time()
svm_linear = SVC(kernel='linear', random_state=RANDOM_STATE)
svm_linear.fit(X_train_sub, y_train_sub)
linear_train_time = time.time() - start

results_linear = evaluate_model(svm_linear, X_test, y_test, "Linear SVM (baseline)")
results_linear['train_time'] = linear_train_time
print(f"Training time: {linear_train_time:.2f}s")


## Step 6: Non-Linear Kernels — RBF and Polynomial

We now train RBF and polynomial kernel SVMs on the same subsample and compare against the
linear baseline. Non-linear kernels can capture curved decision boundaries that a straight
hyperplane cannot, which matters if fraud/legit classes overlap in feature space (as seen in EDA).


In [ ]:
start = time.time()
svm_rbf = SVC(kernel='rbf', random_state=RANDOM_STATE)
svm_rbf.fit(X_train_sub, y_train_sub)
rbf_train_time = time.time() - start

results_rbf = evaluate_model(svm_rbf, X_test, y_test, "RBF SVM")
results_rbf['train_time'] = rbf_train_time
print(f"Training time: {rbf_train_time:.2f}s")


In [ ]:
start = time.time()
svm_poly = SVC(kernel='poly', degree=3, random_state=RANDOM_STATE)
svm_poly.fit(X_train_sub, y_train_sub)
poly_train_time = time.time() - start

results_poly = evaluate_model(svm_poly, X_test, y_test, "Polynomial SVM (degree=3)")
results_poly['train_time'] = poly_train_time
print(f"Training time: {poly_train_time:.2f}s")


### 6.1 Kernel Comparison Table


In [ ]:
comparison_df = pd.DataFrame([results_linear, results_rbf, results_poly])
comparison_df = comparison_df[['model', 'precision', 'recall', 'f1', 'pr_auc', 'train_time']]
comparison_df.columns = ['Kernel', 'Precision', 'Recall', 'F1-Score', 'PR-AUC', 'Training Time (s)']
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
comparison_df.set_index('Kernel')[['Precision', 'Recall', 'F1-Score', 'PR-AUC']].plot(kind='bar', ax=ax)
plt.title('Kernel Comparison: Precision / Recall / F1 / PR-AUC')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


**Observation to write after running:** Compare which kernel gives the best PR-AUC
(the most informative metric for this rare-event problem) versus training time. Usually RBF
performs best for fraud-type data but costs more to train than linear; polynomial can sometimes
underperform if the degree doesn't match the true decision boundary shape.


## Step 7: Stratified Cross-Validation

Before committing to a kernel, we validate performance with **Stratified K-Fold** cross-validation
so results aren't dependent on one lucky split. Stratification ensures every fold preserves the
same (tiny) fraud ratio as the full dataset — plain K-Fold could leave some folds with almost no
fraud examples, which would badly skew the CV scores.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Use the best-performing kernel from Step 6 (update 'rbf' below if a different kernel won)
best_kernel = 'rbf'

cv_scores_f1 = cross_val_score(
    SVC(kernel=best_kernel, random_state=RANDOM_STATE),
    X_train_sub, y_train_sub, cv=skf, scoring='f1'
)

print(f"Stratified 5-Fold F1 scores ({best_kernel} kernel):", cv_scores_f1)
print(f"Mean F1: {cv_scores_f1.mean():.4f}  (+/- {cv_scores_f1.std():.4f})")


## Step 8: Hyperparameter Tuning — GridSearchCV

We tune two key SVM hyperparameters:

- **C (regularization):** Controls the trade-off between a wide margin and correctly classifying
  every training point.
  - Low C → wider margin, tolerates some misclassified points → simpler boundary, more bias, less variance (less overfitting risk).
  - High C → narrower margin, tries to classify every point correctly → complex boundary, less bias, more variance (higher overfitting risk).

- **gamma (RBF kernel coefficient):** Controls how far a single training example's influence reaches.
  - Low gamma → far-reaching influence → smoother, simpler decision boundary (more bias).
  - High gamma → very local influence → boundary hugs individual points tightly (more variance, overfitting risk).

We search a small grid to keep runtime reasonable, using the same subsample and stratified CV.


In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1],
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    SVC(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

start = time.time()
grid_search.fit(X_train_sub, y_train_sub)
grid_search_time = time.time() - start

print(f"\nGrid search took {grid_search_time:.2f}s")
print("Best parameters:", grid_search.best_params_)
print("Best CV F1 score:", grid_search.best_score_)


In [ ]:
best_svm = grid_search.best_estimator_
results_tuned = evaluate_model(best_svm, X_test, y_test, "Tuned RBF SVM (GridSearchCV)")


**Observation to write after running:** Compare the tuned model's metrics against the
un-tuned RBF baseline from Step 6. Note whether the best `C`/`gamma` combination leans toward a
wider or narrower margin, and connect that back to the bias-variance tradeoff explained above.


## Step 9: Feature Selection

We apply **correlation-based filtering**: keep only features with a meaningful correlation to
the target (`Class`), dropping ones that contribute little signal. This reduces dimensionality,
speeds up training, and can reduce noise-driven overfitting.

We then retrain our tuned SVM configuration on this reduced feature set and compare it against
the full feature set on both performance and training time.


In [ ]:
corr_with_target = X_train_sub.corrwith(y_train_sub).abs().sort_values(ascending=False)
print("Feature correlation with Class (absolute value, sorted):\n")
print(corr_with_target)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
corr_with_target.plot(kind='barh', ax=ax)
ax.set_title('Absolute Correlation of Each Feature with Fraud Class')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Keep features above a correlation threshold
CORR_THRESHOLD = 0.05
selected_features = corr_with_target[corr_with_target > CORR_THRESHOLD].index.tolist()

print(f"Selected {len(selected_features)} out of {X_train_sub.shape[1]} features "
      f"(threshold = {CORR_THRESHOLD}):")
print(selected_features)


In [ ]:
X_train_sub_reduced = X_train_sub[selected_features]
X_test_reduced = X_test[selected_features]

start = time.time()
svm_reduced = SVC(**grid_search.best_params_, random_state=RANDOM_STATE)
svm_reduced.fit(X_train_sub_reduced, y_train_sub)
reduced_train_time = time.time() - start

results_reduced = evaluate_model(svm_reduced, X_test_reduced, y_test,
                                  "Tuned RBF SVM (Reduced Features)")
results_reduced['train_time'] = reduced_train_time
print(f"Training time: {reduced_train_time:.2f}s")


### 9.1 Full Feature Set vs. Reduced Feature Set


In [ ]:
results_tuned['train_time'] = grid_search_time  # approx, includes grid search overhead
fs_comparison = pd.DataFrame([results_tuned, results_reduced])
fs_comparison = fs_comparison[['model', 'precision', 'recall', 'f1', 'pr_auc', 'train_time']]
fs_comparison.columns = ['Configuration', 'Precision', 'Recall', 'F1-Score', 'PR-AUC', 'Training Time (s)']
fs_comparison


**Observation to write after running:** Note whether the reduced feature set achieves
comparable PR-AUC/F1 with less training time — if so, that's a strong case for using it in a
real-time production system where inference/training speed matters. If performance drops
noticeably, that signals the dropped features actually carried useful fraud signal.


## Step 10: Final Model Evaluation

We take our final chosen configuration (tuned RBF SVM, reduced feature set — or update this if
your results favored the full feature set) and evaluate it in full: confusion matrix and
precision-recall curve.


In [ ]:
final_model = svm_reduced          # update if full-feature model performed better
final_X_test = X_test_reduced      # update accordingly

y_pred_final = final_model.predict(final_X_test)
y_scores_final = final_model.decision_function(final_X_test)

print(classification_report(y_test, y_pred_final, target_names=['Legitimate', 'Fraud']))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Final Model')
plt.show()


In [ ]:
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_scores_final)
pr_auc_final = auc(rec_curve, prec_curve)

plt.figure(figsize=(7, 5))
plt.plot(rec_curve, prec_curve, label=f'PR-AUC = {pr_auc_final:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Final Model')
plt.legend()
plt.grid(True)
plt.show()


### 10.1 Cost of False Positives vs. False Negatives

Looking at the confusion matrix above:

- **False Positives (legitimate transactions flagged as fraud):** These block real customers
  mid-purchase. Cost = lost sales, customer frustration, support tickets, and merchant
  dissatisfaction (a merchant seeing many of their good customers blocked will lose trust in the
  gateway itself).

- **False Negatives (fraud transactions missed):** These directly cost money — chargebacks, fees,
  and potential card-network penalties for the merchant. Beyond direct loss, repeated missed fraud
  damages the payment gateway's reputation with merchants and card networks.

**Write in your own words after running:** State the actual FP and FN counts from your confusion
matrix, and explain which error type your final model favors avoiding (based on precision vs.
recall values), and whether that tradeoff makes sense for a payment gateway. In production, this
tradeoff is usually tuned further by adjusting the classification threshold on `decision_function`
output rather than just using the default 0.5 boundary — you can mention this as a next step.


## Step 11: Overall Comparison Table (All Configurations)


In [ ]:
all_results = pd.DataFrame([results_linear, results_rbf, results_poly, results_tuned, results_reduced])
all_results = all_results[['model', 'precision', 'recall', 'f1', 'pr_auc', 'train_time']]
all_results.columns = ['Configuration', 'Precision', 'Recall', 'F1-Score', 'PR-AUC', 'Training Time (s)']
all_results


## Step 12: Written Recommendation (150–250 words)

*(Fill this in after you've run all cells and have your actual numbers — replace the placeholder
text below with your own reasoning, referencing your specific PR-AUC, precision, recall and
training time values.)*

> **Draft to edit:**
>
> Based on the kernel comparison, cross-validation, and feature selection results, I recommend
> deploying the **[tuned RBF SVM with reduced feature set / full feature set — pick based on your
> actual numbers]** for production fraud detection.
>
> The RBF kernel outperformed the linear baseline on PR-AUC ([X] vs [Y]), confirming that fraud
> and legitimate transactions are not linearly separable in this feature space — consistent with
> the overlapping distributions observed during EDA. After hyperparameter tuning via
> GridSearchCV (C=[X], gamma=[Y]), the model achieved a precision of [X] and recall of [Y] on the
> held-out test set.
>
> Feature selection reduced the feature count from [X] to [Y] with [minimal/some] impact on
> PR-AUC, while cutting training time by roughly [X]%. For a real-time payment gateway processing
> thousands of transactions per minute, this speed gain is operationally significant, even if it
> costs a small amount of recall.
>
> Given the asymmetric cost of errors — a false positive frustrates a legitimate customer, while
> a false negative results in direct financial loss via chargebacks — I would tune the
> classification threshold to favor **[recall/precision]**, and pair this model with a secondary
> manual review queue for borderline-confidence transactions rather than a single hard cutoff.


---
## Summary of What This Notebook Covered

1. EDA quantifying class imbalance and feature separability
2. Feature scaling and stratified train/test split
3. Baseline Linear SVM
4. RBF and Polynomial kernel comparison
5. Stratified K-Fold cross-validation
6. GridSearchCV tuning of C and gamma
7. Correlation-based feature selection with before/after comparison
8. Final evaluation with confusion matrix, PR curve, and FP/FN cost discussion
9. Production deployment recommendation
